In [ ]:
import os
import sys
import json
import csv
import time
import heapq
import random
import hashlib
import platform
import warnings
from collections import deque, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import binary_dilation, generate_binary_structure
from tqdm import tqdm

warnings.filterwarnings('ignore')

# 1. КОНФИГУРАЦИЯ
CONFIG = {
    "grid_size": 30,
    "patch_size": 30,
    "densities": [0.10, 0.20, 0.30, 0.40, 0.50],
    "maps_per_density": 500,
    "max_attempts_per_density": 10000,   
    "seed": 42,
    "device": "cpu",
    "cnn_epochs": 10,
    "batch_size": 256,
    "lr": 1e-3,
    "warmup_runs": 3,
    "measurement_runs": 3,
    "split": {"type": "stratified", "train_frac": 0.8},  
    "weighted_astar": {"w": 1.05},                      
    "bonferroni_m": 3,                                   
    "bonferroni_alpha": 0.05,                             
    "run_id": time.strftime("run_%Y%m%d_%H%M%S"),         
}

STRUCT = generate_binary_structure(2, 1)  

# 2. ДЕТЕРМИНИЗМ  
def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

def get_env_info():
    return {
        "os": f"{platform.system()} {platform.release()}",
        "processor": platform.processor(),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "numpy_version": np.__version__,
        "scipy_version": __import__("scipy").__version__,
        "run_id": CONFIG["run_id"],
        "seeds": {"numpy_rng": CONFIG["seed"], "torch": CONFIG["seed"], "random": CONFIG["seed"]},
        "determinism": "num_workers=0, torch.use_deterministic_algorithms(True)",
        "baseline": "Euclidean A* (cnn_model=None); cost_opt = backward Dijkstra",
        "note": "Single-threaded CPU is a reproducibility condition, not edge HW simulation.",
    }

# 3. ГЕНЕРАЦИЯ И ГРАФ 
def generate_map(size, density, rng):
    noise = rng.random((size, size)) < (density * 0.5)
    seeds = rng.random((size, size)) < (density * 0.1)
    clusters = binary_dilation(seeds, structure=STRUCT, iterations=3)  # [FIX 3] явный STRUCT
    grid = (noise | clusters).astype(np.uint8)
    grid[0, 0] = 0
    grid[size - 1, size - 1] = 0
    return grid

def get_neighbors_8way_strict(grid, current):
    x, y = current
    size = grid.shape[0]
    out = []
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            if dx == 0 and dy == 0:
                continue
            nx, ny = x + dx, y + dy
            if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                if dx != 0 and dy != 0:
                    if grid[x + dx, y] == 1 or grid[x, y + dy] == 1:
                        continue
                cost = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                out.append(((nx, ny), cost))
    return out

def is_reachable_strict(grid, start, goal):
    size = grid.shape[0]
    visited = np.zeros((size, size), dtype=bool)
    q = deque([start]); visited[start] = True
    while q:
        cur = q.popleft()
        if cur == goal:
            return True
        for nb, _ in get_neighbors_8way_strict(grid, cur):
            if not visited[nb]:
                visited[nb] = True
                q.append(nb)
    return False

def compute_dijkstra_heuristic_map(grid, goal):
    size = grid.shape[0]
    h = np.full((size, size), np.inf)
    h[goal] = 0
    pq = [(0, goal)]
    while pq:
        cost, (x, y) = heapq.heappop(pq)
        if cost > h[x, y]:
            continue
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                if dx == 0 and dy == 0:
                    continue
                nx, ny = x + dx, y + dy
                if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                    if dx != 0 and dy != 0:
                        if grid[x + dx, y] == 1 or grid[x, y + dy] == 1:
                            continue
                    mc = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                    nc = cost + mc
                    if nc < h[nx, ny]:
                        h[nx, ny] = nc
                        heapq.heappush(pq, (nc, (nx, ny)))
    return h

def octile(a, b):
    dx = abs(a[0] - b[0]); dy = abs(a[1] - b[1])
    return (dx + dy) + (np.sqrt(2) - 2) * min(dx, dy)

def sha1_grid(grid):
    return hashlib.sha1(np.ascontiguousarray(grid, dtype=np.uint8).tobytes()).hexdigest()

# 4. МОДЕЛЬ И ДАТАСЕТ
class HeuristicDataset(Dataset):
    def __init__(self, maps, h_stars, patch_size):
        self.samples = []
        pad = patch_size // 2
        for grid, h_star in zip(maps, h_stars):
            size = grid.shape[0]
            goal = (size - 1, size - 1)
            y_c, x_c = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
            max_dist = np.sqrt(2 * (patch_size ** 2))
            dist_goal = np.clip(np.sqrt((x_c - goal[1]) ** 2 + (y_c - goal[0]) ** 2) / max_dist, 0, 1)
            pg = np.pad(grid, pad, constant_values=1)
            pd = np.pad(dist_goal, pad, constant_values=1.0)
            for x in range(size):
                for y in range(size):
                    if grid[x, y] == 0 and h_star[x, y] != np.inf:
                        self.samples.append((
                            np.stack([pg[x:x + patch_size, y:y + patch_size],
                                      pd[x:x + patch_size, y:y + patch_size]], axis=0),
                            h_star[x, y]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        x, y = self.samples[i]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 16, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        return self.fc(torch.flatten(self.pool(x), 1))

# 5. A* С ПРОФИЛИРОВАНИЕМ 
def astar_profiled(grid, start, goal, cnn_model=None, patch_size=30, mode="baseline"):
    """
    mode: "baseline"  — евклидов A* (f=g+h)
          "lh"        — A* + CNN-эвристика (h = max(h_cnn, h_eucl))
          "weighted"  — Weighted A* с Octile и весом w
    Возвращает stats, включая path (список (x,y)).
    """
    size = grid.shape[0]
    t_total_start = time.perf_counter()          

    #  подготовка карт (только для LH)  [FIX 4] внутри t_total_full
    t_prep_start = time.perf_counter()
    pad = patch_size // 2
    padded_grid = padded_dist = None
    if mode == "lh":
        padded_grid = np.pad(grid, pad, constant_values=1)
        y_c, x_c = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
        max_dist = np.sqrt(2 * (patch_size ** 2))
        dist_goal = np.clip(np.sqrt((x_c - goal[1]) ** 2 + (y_c - goal[0]) ** 2) / max_dist, 0, 1)
        padded_dist = np.pad(dist_goal, pad, constant_values=1.0)
    t_prepare_ms = (time.perf_counter() - t_prep_start) * 1000

    open_set = [(0.0, 0.0, 0, start)]
    g_score = {start: 0.0}
    came_from = {}
    counter = 1
    h_cache = {} if mode == "lh" else None

    n_expanded = 0
    n_cnn_calls = 0
    n_cache_hits = 0
    t_infer_ms = 0.0
    t_heap_ms = 0.0                          
    w = CONFIG["weighted_astar"]["w"]

    while open_set:
        t0 = time.perf_counter()
        f, g, _, current = heapq.heappop(open_set)
        t_heap_ms += (time.perf_counter() - t0) * 1000

        if current == goal:
            break
        if g > g_score.get(current, float('inf')):
            continue
        n_expanded += 1

        for nb, cost in get_neighbors_8way_strict(grid, current):
            tg = g_score[current] + cost
            if tg >= g_score.get(nb, float('inf')):
                continue
            g_score[nb] = tg
            came_from[nb] = current

            if mode == "lh":
                if nb in h_cache:
                    h_cnn = h_cache[nb]; n_cache_hits += 1
                else:
                    px, py = nb
                    po = padded_grid[px:px + patch_size, py:py + patch_size]
                    pdd = padded_dist[px:px + patch_size, py:py + patch_size]
                    tin = torch.tensor(np.stack([po, pdd], 0), dtype=torch.float32).unsqueeze(0)
                    ti = time.perf_counter()
                    with torch.no_grad():
                        h_cnn = cnn_model(tin).item()
                    t_infer_ms += (time.perf_counter() - ti) * 1000
                    n_cnn_calls += 1
                    h_cache[nb] = h_cnn
                h_eucl = float(np.hypot(nb[0] - goal[0], nb[1] - goal[1]))
                h_used = max(h_cnn, h_eucl)
            elif mode == "weighted":
                h_used = w * octile(nb, goal)
            else:  # baseline
                h_used = float(np.hypot(nb[0] - goal[0], nb[1] - goal[1]))

            t0 = time.perf_counter()
            heapq.heappush(open_set, (tg + h_used, tg, counter, nb))
            t_heap_ms += (time.perf_counter() - t0) * 1000
            counter += 1

    # восстановление пути  
    t0 = time.perf_counter()
    if current == goal:
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        path.reverse()
        path_cost = g_score[goal]
        success = True
    else:
        path, path_cost, success = [], -1.0, False
    t_path_ms = (time.perf_counter() - t0) * 1000

    t_total_full_ms = (time.perf_counter() - t_total_start) * 1000  # [FIX 4]
    t_other_ms = max(0.0, t_total_full_ms - t_prepare_ms - t_infer_ms - t_heap_ms - t_path_ms)  # [FIX 4]

    return {
        "mode": mode,
        "t_total_full_ms": t_total_full_ms,
        "t_prepare_ms": t_prepare_ms,
        "t_infer_ms": t_infer_ms,
        "t_heap_ms": t_heap_ms,
        "t_path_ms": t_path_ms,
        "t_other_ms": t_other_ms,
        "n_expanded": n_expanded,
        "n_cnn_calls": n_cnn_calls,
        "n_cache_hits": n_cache_hits,
        "path_cost": path_cost,
        "success": success,
        "path": path,
    }

# 6. ПРОВЕРКА МАРШРУТА  
def validate_path(grid, path, start, goal, cost_reported):
    errs = []
    if not path:
        return ["empty_path"], None
    if path[0] != start: errs.append("start_mismatch")
    if path[-1] != goal: errs.append("goal_mismatch")
    recomputed = 0.0
    for (x, y), (nx, ny) in zip(path, path[1:]):
        dx, dy = nx - x, ny - y
        if max(abs(dx), abs(dy)) != 1:
            errs.append(f"non_adjacent_{x}_{y}_{nx}_{ny}"); continue
        if grid[nx, ny] != 0:
            errs.append(f"collision_{nx}_{ny}")
        if dx != 0 and dy != 0:
            if grid[x + dx, y] == 1 or grid[x, y + dy] == 1:
                errs.append(f"corner_cut_{x}_{y}_{nx}_{ny}")
            recomputed += 1.41421356
        else:
            recomputed += 1.0
    if cost_reported >= 0 and abs(recomputed - cost_reported) > 1e-6:
        errs.append("cost_mismatch")
    return errs, recomputed

# 7. ДИАГНОСТИКА h_cnn vs h_star  
def diagnose_heuristic(cnn, grid, h_star, patch_size, tol=1e-6):
    size = grid.shape[0]
    pad = patch_size // 2
    padded_grid = np.pad(grid, pad, constant_values=1)
    y_c, x_c = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    max_dist = np.sqrt(2 * (patch_size ** 2))
    dist_goal = np.clip(np.sqrt((x_c - (size - 1)) ** 2 + (y_c - (size - 1)) ** 2) / max_dist, 0, 1)
    padded_dist = np.pad(dist_goal, pad, constant_values=1.0)
    errs = []
    for x in range(size):
        for y in range(size):
            if grid[x, y] == 0 and h_star[x, y] != np.inf:
                po = padded_grid[x:x + patch_size, y:y + patch_size]
                pdd = padded_dist[x:x + patch_size, y:y + patch_size]
                tin = torch.tensor(np.stack([po, pdd], 0), dtype=torch.float32).unsqueeze(0)
                with torch.no_grad():
                    hc = cnn(tin).item()
                errs.append(hc - h_star[x, y])
    errs = np.array(errs)
    if errs.size == 0:
        return {"freq_over": None, "mae": None, "freq_under": None, "n_cells": 0}
    return {
        "freq_over": float(np.mean(errs > tol)),
        "freq_under": float(np.mean(errs < -tol)),
        "mae": float(np.mean(np.abs(errs))),
        "n_cells": int(errs.size),
    }

# 8. ГЛАВНЫЙ ЦИКЛ 
def run_experiment():
    set_determinism(CONFIG["seed"])  
    os.makedirs("output_data", exist_ok=True)
    print("=== RUN", CONFIG["run_id"], "===")
    print(json.dumps(get_env_info(), indent=2))

    rng = np.random.default_rng(CONFIG["seed"])
    start, goal = (0, 0), (CONFIG["grid_size"] - 1, CONFIG["grid_size"] - 1)

    all_train_maps, all_train_h = [], []
    test_pools = {d: [] for d in CONFIG["densities"]}
    gen_log = {}
    map_registry = [] 

    #  генерация  
    print("\n[1/5] Генерация карт...")
    for d in CONFIG["densities"]:
        attempts = 0
        reachable = 0
        occ_vals = []
        train_n = test_n = 0
        # сначала собираем все достижимые, потом стратифицированно делим
        collected = []
        while reachable < CONFIG["maps_per_density"] and attempts < CONFIG["max_attempts_per_density"]:
            attempts += 1
            grid = generate_map(CONFIG["grid_size"], d, rng)
            if not is_reachable_strict(grid, start, goal):
                continue
            h_star = compute_dijkstra_heuristic_map(grid, goal)
            if h_star[start] == np.inf:
                continue
            reachable += 1
            occ_vals.append(float(grid.mean()))
            collected.append((grid, h_star))
        #  стратифицированное детерминированное деление 80/20
        n = len(collected)
        n_train = int(round(n * CONFIG["split"]["train_frac"]))
        idx = rng.permutation(n)
        train_idx = set(idx[:n_train].tolist())
        for i, (grid, h_star) in enumerate(collected):
            rec = {"density": d, "sha1": sha1_grid(grid),
                   "split": "train" if i in train_idx else "test"}
            map_registry.append(rec)
            if i in train_idx:
                all_train_maps.append(grid); all_train_h.append(h_star); train_n += 1
            else:
                test_pools[d].append((grid, h_star[start], h_star, rec["sha1"])); test_n += 1
        gen_log[d] = {
            "attempts": attempts, "reachable": reachable,
            "train": train_n, "test": test_n,
            "mean_occupancy": float(np.mean(occ_vals)) if occ_vals else None,
        }
        print(f"  density={d:.2f}: attempts={attempts}, reachable={reachable}, "
              f"train={train_n}, test={test_n}, mean_occ={gen_log[d]['mean_occupancy']}")

    # --- обучение  
    print("\n[2/5] Обучение CNN...")
    ds = HeuristicDataset(all_train_maps, all_train_h, CONFIG["patch_size"])
    g_torch = torch.Generator(); g_torch.manual_seed(CONFIG["seed"])
    dl = DataLoader(ds, batch_size=CONFIG["batch_size"], shuffle=True,
                    generator=g_torch, num_workers=0)
    cnn = LightweightCNN().to(CONFIG["device"])
    opt = optim.Adam(cnn.parameters(), lr=CONFIG["lr"])
    crit = nn.MSELoss()
    cnn.train()
    for ep in range(CONFIG["cnn_epochs"]):
        tot = 0.0
        for xb, yb in dl:
            opt.zero_grad()
            loss = crit(cnn(xb).squeeze(), yb)
            loss.backward(); opt.step()
            tot += loss.item()
        print(f"  epoch {ep+1}/{CONFIG['cnn_epochs']} loss={tot/len(dl):.4f}")
    cnn.eval()
    torch.save(cnn.state_dict(), f"output_data/cnn_weights_{CONFIG['run_id']}.pth")  # [FIX 6]

    #  warmup
    print("\n[3/5] Warmup...")
    dummy = torch.randn(1, 2, CONFIG["patch_size"], CONFIG["patch_size"])
    for _ in range(CONFIG["warmup_runs"]):
        with torch.no_grad():
            _ = cnn(dummy)

    #  тесты  
    print("\n[4/5] Тестирование...")
    records = []
    rng_order = random.Random(CONFIG["seed"] + 1)        
    for d in CONFIG["densities"]:
        print(f"  density={d:.2f} (N={len(test_pools[d])})")
        for map_idx, (grid, cost_opt, h_star, sha1) in enumerate(tqdm(test_pools[d], leave=False)):
            reps = {"baseline": [], "lh": [], "weighted": []}
            for _ in range(CONFIG["measurement_runs"]):
                # [FIX 6] детерминированный порядок через rng_order
                order = ["baseline", "lh", "weighted"]
                rng_order.shuffle(order)
                for mode in order:
                    if mode == "lh":
                        s = astar_profiled(grid, start, goal, cnn_model=cnn, mode="lh")
                    elif mode == "weighted":
                        s = astar_profiled(grid, start, goal, mode="weighted")
                    else:
                        s = astar_profiled(grid, start, goal, mode="baseline")
                    reps[mode].append(s)
            # медианы компонент считаем по повторам на уровне карты  
            def med(lst, k): return float(np.median([x[k] for x in lst]))
            b, l, w = reps["baseline"], reps["lh"], reps["weighted"]
            # валидация пути LH 
            lh_path = l[0]["path"]
            errs, recomputed = validate_path(grid, lh_path, start, goal, l[0]["path_cost"])
            records.append({
                "run_id": CONFIG["run_id"], "density": d, "map_id": map_idx, "sha1": sha1,
                "cost_opt": float(cost_opt),
                # baseline
                "t_base_total_ms": med(b, "t_total_full_ms"),
                "n_base_expanded": int(med(b, "n_expanded")),
                # LH
                "t_lh_total_ms": med(l, "t_total_full_ms"),
                "t_lh_prepare_ms": med(l, "t_prepare_ms"),
                "t_lh_infer_ms": med(l, "t_infer_ms"),
                "t_lh_heap_ms": med(l, "t_heap_ms"),
                "t_lh_path_ms": med(l, "t_path_ms"),
                "t_lh_other_ms": med(l, "t_other_ms"),
                "n_lh_expanded": int(med(l, "n_expanded")),
                "n_lh_cnn_calls": int(med(l, "n_cnn_calls")),
                "n_lh_cache_hits": int(med(l, "n_cache_hits")),
                "cost_lh": med(l, "path_cost"),
                "success_lh": int(med(l, "success")),
                "path_valid_lh": int(len(errs) == 0),
                "path_errors_lh": ";".join(errs),
                "path_cost_recomputed_lh": recomputed,
                # Weighted  [FIX 1]
                "t_wa_total_ms": med(w, "t_total_full_ms"),
                "n_wa_expanded": int(med(w, "n_expanded")),
                "cost_wa": med(w, "path_cost"),
                "success_wa": int(med(w, "success")),
            })

    csv_path = f"output_data/final_results_{CONFIG['run_id']}.csv"
    with open(csv_path, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=records[0].keys())
        wr.writeheader(); wr.writerows(records)
    print(f"\nCSV: {csv_path}")

    print("\n[5/5] Диагностика h_cnn vs h_star...")
    diag_rows = []
    for d in CONFIG["densities"]:
        for map_idx, (grid, cost_opt, h_star, sha1) in enumerate(test_pools[d]):
            diag = diagnose_heuristic(cnn, grid, h_star, CONFIG["patch_size"])
            diag.update({"density": d, "map_id": map_idx, "sha1": sha1})
            diag_rows.append(diag)
    with open(f"output_data/diagnostics_{CONFIG['run_id']}.csv", "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=diag_rows[0].keys())
        wr.writeheader(); wr.writerows(diag_rows)

    meta = {
        "run_id": CONFIG["run_id"], "config": CONFIG,
        "env": get_env_info(),
        "architecture": "Conv(2,16,3,p1)-ReLU-Conv(16,32,3,p1)-ReLU-Conv(32,16,3,p1)-ReLU-AdaptiveAvgPool(1)-FC(16,1)",
        "structuring_element": "generate_binary_structure(2,1) (cross, 4-conn), iterations=3",
        "generation_log": {str(k): v for k, v in gen_log.items()},
        "map_registry": map_registry,
    }
    with open(f"output_data/metadata_{CONFIG['run_id']}.json", "w") as f:
        json.dump(meta, f, indent=2)
    print("Артефакты сохранены.")
    return records, gen_log

# 9. АВТО-ТАБЛИЦЫ 
def build_tables(csv_path, gen_log_path):
    rows = list(csv.DictReader(open(csv_path)))
    by_d = defaultdict(list)
    for r in rows: by_d[float(r["density"])].append(r)

    print("\n ТАБЛИЦА 1: ГЕНЕРАЦИЯ ")
    print("| density | attempts | reachable | train | test | mean_occ |")
    for d in sorted(gen_log_path):
        g = gen_log_path[d]
        occ = f"{100*g['mean_occupancy']:.1f}%" if g["mean_occupancy"] is not None else "— (пусто)"
        print(f"| {d:.2f} | {g['attempts']} | {g['reachable']} | {g['train']} | {g['test']} | {occ} |")

    print("\n ТАБЛИЦА 2: ВРЕМЯ ")
    print("| density | N | median T_base | median T_lh | median ratio |")
    for d in sorted(by_d):
        rs = by_d[d]
        tb = np.median([float(r["t_base_total_ms"]) for r in rs])
        tl = np.median([float(r["t_lh_total_ms"]) for r in rs])
        ratio = np.median([float(r["t_lh_total_ms"])/float(r["t_base_total_ms"]) for r in rs])
        print(f"| {d:.2f} | {len(rs)} | {tb:.2f} | {tl:.2f} | {ratio:.2f} |")

    print("\n ТАБЛИЦА 3: РАСКРЫТИЯ ")
    print("| density | mean N_base | mean N_lh | growth % |")
    for d in sorted(by_d):
        rs = by_d[d]
        nb = np.mean([float(r["n_base_expanded"]) for r in rs])
        nl = np.mean([float(r["n_lh_expanded"]) for r in rs])
        print(f"| {d:.2f} | {nb:.2f} | {nl:.2f} | {100*(nl-nb)/nb:.1f}% |")

    print("\n ТАБЛИЦА 4: КАЧЕСТВО (cost_opt = backward Dijkstra) ")
    print("| density | N | median ratio | max ratio (map) | #>1.05 | #>1.0001 | invalid paths |")
    for d in sorted(by_d):
        rs = by_d[d]
        ratios = [(float(r["cost_lh"])/float(r["cost_opt"]), r["map_id"]) for r in rs if float(r["cost_opt"])>0]
        med = np.median([x for x,_ in ratios])
        mx, worst = max(ratios, key=lambda t: t[0])
        n105 = sum(1 for x,_ in ratios if x > 1.05)
        n1 = sum(1 for x,_ in ratios if x > 1.0001)
        bad = sum(1 for r in rs if int(r["path_valid_lh"]) == 0)
        print(f"| {d:.2f} | {len(rs)} | {med:.4f} | {mx:.4f} (map {worst}) | {n105} | {n1} | {bad} |")

# 10. АНАЛИЗ  
def analyze(records, gen_log):
    m = CONFIG["bonferroni_m"]; alpha = CONFIG["bonferroni_alpha"]
    alpha_ind = alpha / m
    lo_p = 100 * (alpha_ind / 2); hi_p = 100 * (1 - alpha_ind / 2)
    print(f"\nBonferroni: m={m}, alpha_ind={alpha_ind:.6f}, "
          f"coverage={100*(1-alpha_ind):.2f}%, percentiles=[{lo_p:.4f}, {hi_p:.4f}]")

    for d in CONFIG["densities"]:
        rs = [r for r in records if float(r["density"]) == d]
        if not rs: continue
        tb = np.mean([float(r["t_base_total_ms"]) for r in rs])
        tl = np.mean([float(r["t_lh_total_ms"]) for r in rs])
        prep = np.mean([float(r["t_lh_prepare_ms"]) for r in rs])
        inf = np.mean([float(r["t_lh_infer_ms"]) for r in rs])
        heap = np.mean([float(r["t_lh_heap_ms"]) for r in rs])
        pth = np.mean([float(r["t_lh_path_ms"]) for r in rs])
        oth = np.mean([float(r["t_lh_other_ms"]) for r in rs])
        nb = np.mean([float(r["n_base_expanded"]) for r in rs])
        nl = np.mean([float(r["n_lh_expanded"]) for r in rs])
        nwa = np.mean([float(r["n_wa_expanded"]) for r in rs])
        print(f"\n--- {d:.2f} (N={len(rs)}) ---")
        print(f"  T_base={tb:.2f} | T_lh={tl:.2f} | T_wa={np.mean([float(r['t_wa_total_ms']) for r in rs]):.2f}")
        print(f"  LH decomp: prep={prep:.2f} inf={inf:.2f} heap={heap:.2f} path={pth:.2f} other={oth:.2f} "
              f"(sum={prep+inf+heap+pth+oth:.2f} vs total={tl:.2f})")
        print(f"  N_base={nb:.2f} | N_lh={nl:.2f} | N_wa={nwa:.2f}")
        c_base_per_expand = tb / nb if nb else float('nan')
        c_lh_per_expand = tl / nl if nl else float('nan')
        print(f"  Условие окупаемости: N_lh/N_base={nl/nb:.3f} < "
              f"c_base/c_lh={c_base_per_expand/c_lh_per_expand:.3f} ? "
              f"{'ДА' if (nl/nb) < (c_base_per_expand/c_lh_per_expand) else 'НЕТ'}")
        ratios = np.array([float(r["cost_lh"])/float(r["cost_opt"]) for r in rs if float(r["cost_opt"])>0])
        if ratios.size:
            print(f"  Quality: median={np.median(ratios):.4f} max={ratios.max():.4f} "
                  f"#>1.05={int((ratios>1.05).sum())} #>1.0001={int((ratios>1.0001).sum())}")

# MAIN
if __name__ == "__main__":
    records, gen_log = run_experiment()
    build_tables(f"output_data/final_results_{CONFIG['run_id']}.csv", gen_log)
    analyze(records, gen_log)
    print("\nГОТОВО:", CONFIG["run_id"])

=== RUN run_20261123_120523 ===
{
  "os": "Darwin 27.2.0",
  "processor": "arm",
  "python_version": "3.13.2",
  "torch_version": "2.14.1",
  "numpy_version": "2.5.4",
  "scipy_version": "1.18.1",
  "run_id": "run_20261123_120523",
  "seeds": {
    "numpy_rng": 42,
    "torch": 42,
    "random": 42
  },
  "determinism": "num_workers=0, torch.use_deterministic_algorithms(True)",
  "baseline": "Euclidean A* (cnn_model=None); cost_opt = backward Dijkstra",
  "note": "Single-threaded CPU is a reproducibility condition, not edge HW simulation."
}

[1/5] Генерация карт...
  density=0.10: attempts=729, reachable=500, train=400, test=100, mean_occ=0.23786888888888888
  density=0.20: attempts=1916, reachable=500, train=400, test=100, mean_occ=0.39497333333333334
  density=0.30: attempts=10000, reachable=322, train=258, test=64, mean_occ=0.4996859903381642
  density=0.40: attempts=10000, reachable=23, train=18, test=5, mean_occ=0.5908212560386472
  density=0.50: attempts=10000, reachable=0, trai

  density=0.20 (N=100)


  density=0.30 (N=64)


  density=0.40 (N=5)


  density=0.50 (N=0)



CSV: output_data/final_results_run_20261123_120523.csv

[5/5] Диагностика h_cnn vs h_star...
Артефакты сохранены.

 ТАБЛИЦА 1: ГЕНЕРАЦИЯ 
| density | attempts | reachable | train | test | mean_occ |
| 0.10 | 729 | 500 | 400 | 100 | 0.2% |
| 0.20 | 1916 | 500 | 400 | 100 | 0.4% |
| 0.30 | 10000 | 322 | 258 | 64 | 0.5% |
| 0.40 | 10000 | 23 | 18 | 5 | 0.6% |
| 0.50 | 10000 | 0 | 0 | 0 | — (пусто) |

 ТАБЛИЦА 2: ВРЕМЯ 
| density | N | median T_base | median T_lh | median ratio |
| 0.10 | 100 | 1.21 | 71.49 | 57.19 |
| 0.20 | 100 | 1.11 | 46.87 | 43.81 |
| 0.30 | 64 | 0.90 | 37.89 | 42.09 |
| 0.40 | 5 | 0.77 | 34.53 | 44.71 |

 ТАБЛИЦА 3: РАСКРЫТИЯ 
| density | mean N_base | mean N_lh | growth % |
| 0.10 | 257.90 | 554.56 | 115.0% |
| 0.20 | 272.63 | 370.34 | 35.8% |
| 0.30 | 237.12 | 285.33 | 20.3% |
| 0.40 | 196.00 | 218.40 | 11.4% |

 ТАБЛИЦА 4: КАЧЕСТВО (cost_opt = backward Dijkstra) 
| density | N | median ratio | max ratio (map) | #>1.05 | #>1.0001 | invalid paths |
| 0.10 | 100 | 1